# Task 1 - Drone Crop Disease Classification

This notebook is a thin runner around the `src/` package (`config.py`, `data.py`,
`model.py`, `train.py`, `evaluate.py`, `inference.py`, `plotting.py`). The same
functions are used here and in `main.py`, so results are identical whether you run
this notebook, run `python main.py` in VS Code, or run it in Colab.

**Setup, either environment:**
- **VS Code / local:** open this repository as your workspace root, create a venv,
  `pip install -r requirements.txt`, then run this notebook with the Jupyter
  extension (kernel = your venv), or just run `python main.py` from a terminal.
- **Colab:** run the cell below first. It detects Colab, and if `src/` is not
  already present in the runtime, prompts you to upload the project zip.


In [1]:
import sys
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    if not os.path.isdir("src"):
        print("Running in Colab and 'src/' was not found in the current directory.")
        print("Upload the project zip now (the same folder this notebook ships in).")
        from google.colab import files
        uploaded = files.upload()
        zip_name = next(iter(uploaded.keys()))
        import zipfile
        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(".")
        # If the zip extracted into a subfolder, move into it.
        for entry in os.listdir("."):
            if os.path.isdir(entry) and os.path.isdir(os.path.join(entry, "src")):
                os.chdir(entry)
                break
    !pip install -q -r requirements.txt
else:
    print("Not running in Colab; assuming a local environment with requirements.txt already installed.")

print("Working directory:", os.getcwd())
print("src/ present:", os.path.isdir("src"))


Not running in Colab; assuming a local environment with requirements.txt already installed.
Working directory: d:\0 AMAAN MAIN\Documents\GERMANY\BSBI\BSBI-Assignments\Sem2\Advanced Robotics Automation and Computer Vision\task1-code\notebook
src/ present: False


## 1. Imports and configuration

In [2]:
# root directory for imports
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "config.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: D:\0 AMAAN MAIN\Documents\GERMANY\BSBI\BSBI-Assignments\Sem2\Advanced Robotics Automation and Computer Vision\task1-code


In [3]:
from src.config import Config, set_seed
from src.data import build_dataloaders, load_plantvillage
from src.model import build_model, count_parameters
from src.train import train_model, save_checkpoint, clear_checkpoint
from src.evaluate import evaluate_model, top_confusions
from src.inference import sample_predictions
from src.plotting import (
    plot_class_distribution, plot_augmented_samples,
    plot_training_curves, plot_confusion_matrix, plot_inference_samples,
)

# Adjust these as needed; Config's defaults are reasonable for a first run.
#
# use_data_parallel: automatically uses both GPUs on a dual-GPU Kaggle
#   session via nn.DataParallel, and quietly falls back to a single GPU
#   (e.g. your local machine) otherwise. No manual switching needed.
#
# enable_thermal_throttle / gpu_high_temp_c / gpu_resume_temp_c: for a
#   local GPU that overheats during long runs. Training pauses once the
#   hottest visible GPU hits gpu_high_temp_c and resumes automatically
#   once it drops to gpu_resume_temp_c. Has no effect if nvidia-smi is
#   unavailable (most Kaggle sessions), so it's safe to leave on always.
#
# resume_from_checkpoint: if a previous run was interrupted (Kaggle
#   disconnect, kernel restart, manual stop), re-running this notebook
#   picks up from the last completed epoch instead of starting over.
#   Set to False, or call clear_checkpoint(cfg) below, to force a fresh run.
cfg = Config( num_epochs=8, batch_size=32, learning_rate=3e-4, output_dir="outputs", use_data_parallel=True, resume_from_checkpoint=True, enable_thermal_throttle=True, gpu_high_temp_c=70, gpu_resume_temp_c=45, )
set_seed(cfg.seed)
print(f"Device: {cfg.device} | GPUs visible: {cfg.num_gpus}")

# Uncomment to force a fresh run instead of resuming from a checkpoint:
# clear_checkpoint(cfg)


Device: cuda | GPUs visible: 1


## 2. Load data

In [4]:
train_loader, val_loader, label_names = build_dataloaders(cfg)
print(f"Classes: {len(label_names)}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

2026-09-14 20:13:11 | INFO     | src.data | Loading dataset mohanty/PlantVillage via direct zip extraction


Casting the dataset:   0%|          | 0/54305 [00:00<?, ? examples/s]

2026-09-14 20:13:12 | INFO     | src.data | Loaded 54305 images across 38 classes
2026-09-14 20:13:16 | INFO     | src.data | Train/val split: 43444 / 10861
Classes: 38
Train batches: 1358, Val batches: 340


In [5]:
raw_train, _, _ = load_plantvillage(cfg)
plot_class_distribution(raw_train, label_names, cfg)
plot_augmented_samples(train_loader, label_names, cfg)


2026-09-14 20:13:17 | INFO     | src.data | Loading dataset mohanty/PlantVillage via direct zip extraction


Casting the dataset:   0%|          | 0/54305 [00:00<?, ? examples/s]

2026-09-14 20:13:17 | INFO     | src.data | Loaded 54305 images across 38 classes
2026-09-14 20:13:21 | INFO     | src.data | Train/val split: 43444 / 10861
2026-09-14 20:13:23 | INFO     | src.plotting | Saved figure to outputs\class_distribution.png
2026-09-14 20:13:36 | INFO     | src.plotting | Saved figure to outputs\augmented_samples.png


WindowsPath('outputs/augmented_samples.png')

## 3. Build model

In [6]:
model = build_model(num_classes=len(label_names), use_data_parallel=cfg.use_data_parallel)
param_counts = count_parameters(model)
print(f"Total parameters: {param_counts['total']:,}")
print(f"Trainable parameters: {param_counts['trainable']:,}")


Total parameters: 2,272,550
Trainable parameters: 2,272,550


## 4. Train

In [8]:
# If this cell is interrupted (Kaggle disconnect, kernel restart, manual
# stop), just re-run this cell (and the two cells above it) after
# reconnecting; a checkpoint is written after every epoch to
# outputs/training_checkpoint.pth and training resumes from the next
# epoch automatically instead of starting over. If your local GPU gets
# too hot, training pauses on its own and resumes once it cools down.
model, history = train_model(model, train_loader, val_loader, cfg)
plot_training_curves(history, cfg)


2026-09-10 18:44:29 | INFO     | src.gpu_monitor | GPU temperature 70°C reached threshold 70°C; pausing training until it drops to 45°C.
2026-09-10 18:44:44 | INFO     | src.gpu_monitor | Cooling down: GPU at 54°C (resume at 45°C)...
2026-09-10 18:44:59 | INFO     | src.gpu_monitor | Cooling down: GPU at 50°C (resume at 45°C)...
2026-09-10 18:45:15 | INFO     | src.gpu_monitor | Cooling down: GPU at 47°C (resume at 45°C)...
2026-09-10 18:45:30 | INFO     | src.gpu_monitor | Cooling down: GPU at 46°C (resume at 45°C)...
2026-09-10 18:45:45 | INFO     | src.gpu_monitor | Cooling down: GPU at 45°C (resume at 45°C)...
2026-09-10 18:45:45 | INFO     | src.gpu_monitor | GPU cooled to 45°C; resuming training.
2026-09-10 18:46:27 | INFO     | src.gpu_monitor | GPU temperature 71°C reached threshold 70°C; pausing training until it drops to 45°C.
2026-09-10 18:46:42 | INFO     | src.gpu_monitor | Cooling down: GPU at 55°C (resume at 45°C)...
2026-09-10 18:46:58 | INFO     | src.gpu_monitor | Coo

WindowsPath('outputs/training_curves.png')

## 5. Evaluate

In [9]:
result = evaluate_model(model, val_loader, label_names, cfg.device)
print(result.report_text)

print("Top confusions (true -> predicted: count):")
for true_label, pred_label, count in top_confusions(result, label_names):
    print(f"  {true_label} -> {pred_label}: {count}")

plot_confusion_matrix(result, label_names, cfg)

2026-09-10 19:32:31 | INFO     | src.evaluate | Evaluation: accuracy=0.9911 macro_precision=0.9864 macro_recall=0.9906 macro_f1=0.9883
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       1.00      1.00      1.00       126
                                 Apple___Black_rot       1.00      1.00      1.00       124
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        55
                                   Apple___healthy       1.00      1.00      1.00       329
                               Blueberry___healthy       1.00      1.00      1.00       301
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       210
                 Cherry_(including_sour)___healthy       1.00      0.99      1.00       171
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.89      0.98      0.93       103
                       Corn_(maize)_

WindowsPath('outputs/confusion_matrix.png')

## 6. Inference samples

In [15]:
predictions = sample_predictions(model, val_loader.dataset, label_names, cfg, cfg.device)
plot_inference_samples(predictions, cfg)

2026-09-10 19:36:38 | INFO     | src.plotting | Saved figure to outputs\inference_samples.png


WindowsPath('outputs/inference_samples.png')

## 7. Save checkpoint and results summary

In [11]:
import json

save_checkpoint(model, label_names, cfg)

summary = {
    "accuracy": result.accuracy,
    "macro_precision": result.macro_precision,
    "macro_recall": result.macro_recall,
    "macro_f1": result.macro_f1,
    "best_val_acc_during_training": history.best_val_acc,
    "training_time_minutes": history.elapsed_seconds / 60,
    "epochs_run": cfg.num_epochs,
    "total_parameters": param_counts["total"],
}
with open(cfg.output_dir / "results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


2026-09-10 19:33:41 | INFO     | src.train | Model saved to outputs\mobilenetv2_plantvillage_drone.pth
{
  "accuracy": 0.9910689623423258,
  "macro_precision": 0.9864453586551422,
  "macro_recall": 0.9906262314309942,
  "macro_f1": 0.9883417259740097,
  "best_val_acc_during_training": 0.9910689623423258,
  "training_time_minutes": 73.57360209623972,
  "epochs_run": 8,
  "total_parameters": 2272550
}


## 8. Persist outputs beyond the Colab session (optional)

If running in Colab, mount Drive to keep the checkpoint, figures and
`results_summary.json` after the runtime disconnects, and to get a
shareable link for the report submission.


In [12]:
if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r outputs /content/drive/MyDrive/task1_outputs
    print("Copied outputs/ to Google Drive at task1_outputs/")


## Notes for the report

`outputs/results_summary.json` contains the exact numbers to quote in report
sections 3.7 and 3.8: accuracy, macro precision/recall/F1, best validation
accuracy during training, training time, epochs run, and parameter count.
The same folder contains all five figures referenced in the report:
`class_distribution.png`, `augmented_samples.png`, `training_curves.png`,
`confusion_matrix.png`, `inference_samples.png`.
